# Text conditioning by cross-attention on CIFAR-100

Baseline 3 of 3: the condition enters the network as text, through attention, with **no
guidance at sampling time**. Trains a base diffusion model (or loads one), freezes it,
adds one cross-attention block at the bottleneck, trains only that block, samples.

**Paper:** Chen, Yu, Ge, Yao, Xie, Wu, Wang, Kwok, Luo, Lu, Li.
*PixArt-α: Fast Training of Diffusion Transformer for Photorealistic Text-to-Image
Synthesis.* ICLR 2024. [arXiv:2310.00426](https://arxiv.org/abs/2310.00426)

**Official code exists** and was read:
[PixArt-alpha/PixArt-alpha](https://github.com/PixArt-alpha/PixArt-alpha),
`MultiHeadCrossAttention` in `PixArt_blocks.py` and its zero-init in `PixArt.py`.
`src/guidance/baselines/cross_attention.py` re-expresses that block in plain PyTorch for
a conv U-Net; its header lists what was kept and what changed.

**What this arm tests, and why the label path is off by default.** The TC-LoRA run showed
that a text embedding of a class name adds nothing on top of the class label — the
adapter moved 11% and changed nothing (`docs/findings.md`). So here the backbone runs
with the **null label** and the text is the *only* route by which the model learns the
class. If cross-attention conditions at all, it has to show up. Set `label_path = True`
to reproduce the TC-LoRA setting instead.

Run top to bottom. On Colab set **Runtime > Change runtime type > GPU** first.

## How this notebook implements the PixArt-α block

**Notation.** $x_k$ is the noisy image at diffusion step $k$ ($K = 1000$; the network receives
$t = k/K$), $\varepsilon_\theta$ the frozen noise-prediction U-Net, $c$ the class, $\varnothing$
the null label, $h(c) \in \mathbb R^{L \times 512}$ the frozen CLIP *token* embeddings of
"a photo of a {class}" with padding mask $M$. $\omega$ are the parameters trained here.

**The paper's idea.** PixArt-α feeds text into a diffusion transformer through a cross-attention
block in every transformer block: the image tokens form the queries, the text tokens the keys
and values, and the result is added back as a plain residual. The block's output projection is
zero-initialised, so a freshly attached block is the identity and training starts from the
unconditioned model:

$$Q = z W_q,\qquad [K\;\;V] = h(c)\, W_{kv},\qquad
\mathrm{Attn}(z, h(c)) = \operatorname{softmax}\!\Big(\frac{Q K^\top}{\sqrt d} + \log M\Big) V$$

$$z' = z + \mathrm{Attn}(z, h(c))\, W_o,\qquad W_o = 0 \text{ at initialisation}$$

**How it is used here.** One such block is attached at the frozen U-Net's bottleneck, where the
feature map $z \in \mathbb R^{256 \times 8 \times 8}$ is flattened to 64 tokens. Four heads. Only
$W_q, W_{kv}, W_o$ train (394k parameters), with the ordinary noise-prediction loss and the null
label, so the text is the model's only route to the class:

$$\mathcal L(\omega) = \mathbb E_{k,\,x_0,\,\varepsilon}\Big[\big\|\varepsilon -
\varepsilon_{\theta,\omega}\big(x_k, k, \varnothing;\ h(c)\big)\big\|_2^2\Big]$$

There is no guidance at sampling: plain DDIM with the conditioned network, $N$ evaluations.
This arm therefore measures what the conditioning mechanism does on its own.

| In PixArt-α (paper and official code) | In this notebook |
|---|---|
| `MultiHeadCrossAttention` in `PixArt_blocks.py`: `q_linear` on image tokens, `kv_linear` on text tokens, one output `proj`, padding mask | `TextCrossAttention` in `cross_attention.py` |
| `proj` zero-initialised in `PixArt.initialize_weights`, so the block starts as the identity | `TextCrossAttention.proj`; the `allclose` assertion in *Attach cross-attention* |
| `x = x + cross_attn(x, y, mask)` in `PixArtBlock.forward`: a plain residual, no norm, no gate | `_Conditioned`, wrapped around the U-Net stage named by `cfg.attach`, inside `TextConditioned` |
| T5-XXL caption tokens through a `y_embedder` MLP | CLIP token embeddings from `build_token_embeddings`, straight into `kv_linear`; no MLP |
| cross-attention in every DiT block | one block, at the U-Net bottleneck (`attach = "mid"`) |
| not in PixArt-α: null label, wrong-text pass, matched validation | `COND`, `xa_val_wrong_text`, `validate`: our evaluation protocol, identical in the TC-LoRA notebook |

**What the paper did that this notebook does not.** Their text encoder is T5-XXL (4096-dim);
ours is CLIP, the same frozen encoder every arm here uses, so the comparison isolates the
mechanism and not the encoder. They place the block in every transformer block and train the
whole 600M-parameter model in stages on 25M images; we attach one block to a frozen 1.9M
U-Net and train only that block on 45k images. They report results with classifier-free guidance
at scale 4.5; we sample with none. Six deviations are listed in the header of
`src/guidance/baselines/cross_attention.py`.

**Why token embeddings and not a pooled vector.** With a single key the softmax is identically 1
and the block collapses to adding a constant. It would still condition, but it could not select
between words, which is the mechanism under test.

In [ ]:
# Colab bootstrap. Run once.
import importlib.util, subprocess, sys, os

for pkg in ("transformers", "datasets"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

if importlib.util.find_spec("guidance") is None:
    # Inside a checkout (Jupyter on a server, cwd somewhere under the repo): walk up to its src/.
    from pathlib import Path as _P
    _src = next((p / "src" for p in [_P.cwd(), *_P.cwd().parents] if (p / "src" / "guidance").is_dir()), None)
    if _src is None:                                   # Colab: clone the repo next to the notebook
        if not os.path.exists("discrete-generative-models-proj-1"):
            subprocess.run(["git", "clone", "-q",
                            "https://github.com/imperorrp/discrete-generative-models-proj-1.git"],
                           check=True)
        _src = "discrete-generative-models-proj-1/src"
    sys.path.insert(0, str(_src))
    print("guidance package from:", _src)

import torch
from guidance.baselines.cross_attention import TextConditioned

def _git_sha():
    for d in ("discrete-generative-models-proj-1", "."):
        try:
            return subprocess.check_output(["git", "-C", d, "rev-parse", "--short", "HEAD"],
                                           text=True, stderr=subprocess.DEVNULL).strip()
        except Exception:
            pass
    return "unknown"

GIT_SHA = _git_sha()
print("code version:", GIT_SHA)
print("device:", "cuda" if torch.cuda.is_available()
      else "CPU  <-- Runtime > Change runtime type > GPU")

## Data source

Nothing to do here. The loader tries a local copy first (`./cifar`, `/content/cifar`,
`/content/drive/MyDrive/cifar`, ...), then the HuggingFace CDN. It never touches
`cs.toronto.edu`, which is rate-limited to ~50 kB/s.

## Config

In [ ]:
import math, time, json
from dataclasses import dataclass, asdict, fields
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class Cfg:
    seed: int = 10                 # training randomness
    split_seed: int = 10           # validation split. Fixed on purpose: change `seed` for a
                                   # multi-seed run WITHOUT moving training images into val.
    batch_size: int = 64
    base_epochs: int = 30
    xattn_epochs: int = 15
    lr_base: float = 2e-4
    lr_xattn: float = 1e-4
    K: int = 1000                  # diffusion training steps
    sample_steps: int = 50         # sampling steps
    eta: float = 0.0               # 0 deterministic DDIM; 1 ancestral DDPM
    heads: int = 4
    attach: str = "mid"            # U-Net stage whose output the text conditions
    label_path: bool = False       # False: backbone sees the null label; text is the only condition
    reuse_base: bool = True        # load base.pt if present instead of retraining
    limit: int = 0                 # >0 truncates the training set, for smoke tests

cfg = Cfg()

# Any field can be overridden from the environment, e.g. CFG_BASE_EPOCHS=1 CFG_LIMIT=5000, so
# headless runs (nbconvert, a server, a different GPU) never need a cell edited.
for _f in fields(Cfg):
    _v = os.environ.get(f"CFG_{_f.name.upper()}")
    if _v is not None:
        _cur = getattr(cfg, _f.name)
        setattr(cfg, _f.name, tuple(float(t) for t in _v.split(",")) if isinstance(_cur, tuple)
                else type(_cur)(_v) if not isinstance(_cur, bool) else _v.lower() in ("1", "true", "yes"))
        print(f"cfg.{_f.name} <- {getattr(cfg, _f.name)!r} (from CFG_{_f.name.upper()})")
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
print(DEV, asdict(cfg))

## Data

From `src/guidance/data/cifar.py`. Pixels normalised to `[-1, 1]`; 50 images per class
held out for validation with a seeded per-class shuffle.

In [ ]:
from guidance.data.cifar import make_loaders

train_loader, val_loader, meta = make_loaders(
    batch_size=cfg.batch_size, seed=cfg.split_seed, limit=cfg.limit)

xb, cb = next(iter(train_loader))
print("x", tuple(xb.shape), f"[{float(xb.min()):.2f}, {float(xb.max()):.2f}]",
      "| cond", tuple(cb.shape))
print("train batches", len(train_loader), "| val batches", len(val_loader))
print("conditions", meta["n_conditions"], "| example text:", meta["cond_text"][0])

## Frozen text encoder — token embeddings, not a pooled vector

The TC-LoRA arm used CLIP's pooled 512-vector. Cross-attention needs the **per-token**
embeddings and the padding mask: with a single key token the softmax is identically 1 and
the block degenerates into adding a constant. It still conditions, but it cannot select
between words, which is the point of the mechanism.

Falls back to random tokens if CLIP will not load; the manifest records that, and a
fallback run is not a result.

In [ ]:
def build_token_embeddings(texts, device):
    '''Returns (tokens (N, L, D), mask (N, L) bool, is_fallback).'''
    try:
        from transformers import CLIPTokenizer, CLIPTextModel, logging as hf_logging
        hf_logging.set_verbosity_error()          # silence the vision_model.* UNEXPECTED report
        name = "openai/clip-vit-base-patch32"
        tok = CLIPTokenizer.from_pretrained(name)
        enc = CLIPTextModel.from_pretrained(name).to(device).eval()
        for p in enc.parameters():
            p.requires_grad_(False)
        b = tok(texts, padding=True, truncation=True, return_tensors="pt")
        with torch.no_grad():
            hidden = enc(**{k: v.to(device) for k, v in b.items()}).last_hidden_state
        return hidden, b["attention_mask"].to(device).bool(), False
    except Exception as e:
        print(f"CLIP unavailable ({type(e).__name__}); using random fallback tokens.")
        g = torch.Generator().manual_seed(0)
        toks = torch.randn(len(texts), 8, 512, generator=g).to(device)
        return toks, torch.ones(len(texts), 8, dtype=torch.bool, device=device), True

TEXT_TOK, TEXT_MASK, IS_FALLBACK = build_token_embeddings(meta["cond_text"], DEV)
TEXT_DIM = TEXT_TOK.shape[-1]
print("token embeddings:", tuple(TEXT_TOK.shape), "| real tokens per prompt:",
      TEXT_MASK.sum(1).tolist()[:5], "... | fallback:", IS_FALLBACK)

## Backbone

In [ ]:
def timestep_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10_000.0) * torch.arange(half, device=t.device) / half)
    args = t.float()[:, None] * freqs[None, :]
    return torch.cat([args.cos(), args.sin()], dim=-1)


def conv_block(cin, cout, groups=8):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1), nn.GroupNorm(groups, cout), nn.SiLU(),
        nn.Conv2d(cout, cout, 3, padding=1), nn.GroupNorm(groups, cout), nn.SiLU(),
    )


class UNet(nn.Module):
    '''Class-conditional U-Net. The label enters as a per-channel bias after block 1.'''

    def __init__(self, c=64, n_classes=100, tdim=128):
        super().__init__()
        self.tdim, self.null = tdim, n_classes
        self.t_mlp = nn.Sequential(nn.Linear(tdim, c), nn.SiLU(), nn.Linear(c, c))
        self.label = nn.Embedding(n_classes + 1, c)      # +1 = null token
        self.e1, self.e2 = conv_block(3, c), conv_block(c, 2 * c)
        self.mid = conv_block(2 * c, 4 * c)
        self.d2, self.d1 = conv_block(6 * c, 2 * c), conv_block(3 * c, c)
        self.out = nn.Conv2d(c, 3, 1)
        self.pool = nn.MaxPool2d(2)
        nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)

    def forward(self, x, t, cond_ids=None):
        if cond_ids is None:
            cond_ids = torch.full((x.shape[0],), self.null, device=x.device, dtype=torch.long)
        bias = (self.t_mlp(timestep_embedding(t, self.tdim))
                + self.label(cond_ids))[:, :, None, None]
        s1 = self.e1(x) + bias
        s2 = self.e2(self.pool(s1))
        h = self.mid(self.pool(s2))
        h = self.d2(torch.cat([F.interpolate(h, size=s2.shape[-2:]), s2], 1))
        h = self.d1(torch.cat([F.interpolate(h, size=s1.shape[-2:]), s1], 1))
        return self.out(h)


def make_backbone():
    return UNet(64, meta["n_conditions"])

torch.manual_seed(cfg.seed)      # model init and training randomness follow `seed`, not the split
net = make_backbone().to(DEV)
MID_CHANNELS = 4 * 64                    # output width of the stage named by cfg.attach
print(f"backbone params: {sum(p.numel() for p in net.parameters()):,}")

## Diffusion — cosine schedule, epsilon prediction, DDIM sampler

In [ ]:
def cosine_alpha_bar(K, s=0.008, device="cpu"):
    steps = torch.arange(K + 1, dtype=torch.float32, device=device) / K
    f = torch.cos((steps + s) / (1 + s) * math.pi / 2) ** 2
    return (f / f[0])[1:].clamp(1e-5, 0.9999)

ABAR = cosine_alpha_bar(cfg.K, device=DEV)
NULL_ID = meta["n_conditions"]

# At the first DDIM step sqrt(abar) is about 0.003, so x0 = (x - sqrt(1-abar) eps)/sqrt(abar)
# divides by a tiny number and any error in eps is amplified ~300x. Without clamping the
# samples blow up. Pixels are normalised to [-1, 1], so the clamp is 1.
CLIP_X0 = 1.0


def add_noise(x0, gen=None):
    '''gen: pass a seeded torch.Generator to get the same (k, eps) every call.'''
    n = x0.shape[0]
    k = torch.randint(0, cfg.K, (n,), device=x0.device, generator=gen)
    eps = torch.randn(x0.shape, device=x0.device, generator=gen)
    ab = ABAR[k].view(n, *([1] * (x0.dim() - 1)))
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps, k, eps


def drop_labels(y, p_drop=0.1):
    '''Condition dropout creates the null branch that guidance needs later.'''
    keep = torch.rand(y.shape[0], device=y.device) >= p_drop
    return torch.where(keep, y, torch.full_like(y, NULL_ID))


@torch.no_grad()
def ddim_sample(eps_fn, n, shape, steps, clip_x0=None, n_snapshots=0, x_init=None, eta=None, generator=None):
    '''eps_fn(x, t) -> predicted noise. Deterministic DDIM.

    Convention, shared by every arm in this project: clip the predicted x0, then recompute
    eps from the clipped x0 before stepping (as TFG's official sampler does). Keeping the
    original eps is a different update whenever the clamp engages.

    x_init: pass the same starting noise to two calls for a paired comparison; with eta > 0
    also pass identically seeded generators, or the per-step noise differs.
    eta: 0 deterministic DDIM (images), 1 ancestral DDPM (cells, as the base's own notebook).
    n_snapshots > 0 also returns a list of (timestep, image) pairs for the trajectory
    figure: the starting noise, then the model's clean-image estimate x0_hat at that many
    evenly spaced steps. The last pair is the finished sample.
    '''
    clip_x0 = CLIP_X0 if clip_x0 is None else clip_x0
    idx = torch.linspace(cfg.K - 1, 0, steps).long().to(DEV)
    keep = set(torch.linspace(1, steps - 1, max(n_snapshots, 1)).long().tolist())
    eta = cfg.eta if eta is None else eta
    x = torch.randn(n, *shape, device=DEV, generator=generator) if x_init is None else x_init.clone()
    snaps = [(cfg.K, x.cpu().clone())] if n_snapshots else []
    for i, k in enumerate(idx):
        t = (k.float() / cfg.K).repeat(n)
        eps = eps_fn(x, t)
        ab = ABAR[k]
        ab_prev = ABAR[idx[i + 1]] if i + 1 < len(idx) else torch.ones_like(ab)
        x0 = (x - (1 - ab).sqrt() * eps) / ab.sqrt()
        if clip_x0:
            x0 = x0.clamp(-clip_x0, clip_x0)
            eps = (x - ab.sqrt() * x0) / (1 - ab).sqrt()      # recompute from the clipped x0
        # eta = 0 is deterministic DDIM; eta = 1 is ancestral DDPM (DDIM paper eq. 16), with fresh
        # noise at every step drawn from `generator` so paired rows stay paired.
        sigma = eta * ((1 - ab_prev) / (1 - ab) * (1 - ab / ab_prev)).sqrt()
        x = ab_prev.sqrt() * x0 + (1 - ab_prev - sigma ** 2).clamp(min=0).sqrt() * eps
        if eta > 0:
            x = x + sigma * torch.randn(x.shape, device=DEV, generator=generator)
        if n_snapshots and i in keep:
            # x0_hat, not x_t: the noisy state looks like noise until the last few steps,
            # while the clean estimate shows what the model believes at every step.
            snaps.append((int(k), x0.cpu().clone()))
    return (x, snaps) if n_snapshots else x

## Base model — train, or load `base.pt` if it is already here

The base is identical to the TC-LoRA arm's. If you have `base.pt` from that run, upload it
to the session and this cell loads it instead of spending eight minutes retraining. Both
arms then share exactly the same frozen weights, which is what makes the comparison a
controlled one.

Training applies 10% label dropout; validation is fully conditional on fixed noise. See the
TC-LoRA notebook for why those differ.

In [ ]:
VAL_SEED = cfg.seed + 999


def run_epochs(model, params, n_epochs, lr, tag, train_fn, val_fn, ckpt):
    '''Trains with train_fn, validates with val_fn on fixed noise, keeps the best checkpoint.'''
    opt = torch.optim.AdamW(params, lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(n_epochs, 1))
    hist, best, best_ep = [], float("inf"), 0
    for ep in range(1, n_epochs + 1):
        model.train(); tot = num = 0; t0 = time.time()
        for x, y in train_loader:
            x, y = x.to(DEV), y.to(DEV)
            loss = train_fn(x, y)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
            tot += loss.item() * x.shape[0]; num += x.shape[0]
        sched.step()

        val = validate(model, val_fn)
        is_best = val < best
        if is_best:
            best, best_ep = val, ep
            torch.save({"state": model.state_dict(), "epoch": ep, "val": val}, ckpt)
        hist.append({"epoch": ep, "train": tot / num, "val": val, "best": is_best,
                     "seconds": time.time() - t0})
        print(f"{tag} epoch {ep:3d}  train {tot/num:.4f}  val {val:.4f}"
              f"{'  *best' if is_best else ''}  ({time.time()-t0:.0f}s)")
    print(f"{tag}: best val {best:.4f} at epoch {best_ep}, saved to {ckpt}")
    return hist


@torch.no_grad()
def validate(model, val_fn):
    '''Same examples, same timesteps, same noise on every call -- for any model.'''
    model.eval()
    gen = torch.Generator(device=DEV).manual_seed(VAL_SEED)
    tot = num = 0
    for x, y in val_loader:
        x, y = x.to(DEV), y.to(DEV)
        tot += val_fn(x, y, gen).item() * x.shape[0]; num += x.shape[0]
    return tot / num


def base_train(x, y):
    xk, k, eps = add_noise(x)
    return F.mse_loss(net(xk, k.float() / cfg.K, drop_labels(y)), eps)


def base_val(x, y, gen):
    xk, k, eps = add_noise(x, gen)
    return F.mse_loss(net(xk, k.float() / cfg.K, y), eps)     # no dropout


# What a base.pt must agree on before it can be reused. A checkpoint trained under a
# different split would put its own training images into this run's validation set.
CKPT_META = {"split_seed": cfg.split_seed, "n_conditions": meta["n_conditions"],
             "shape": list(meta["shape"]), "K": cfg.K, "schedule": "cosine",
             "backbone_params": sum(p.numel() for p in net.parameters())}

if cfg.reuse_base and os.path.exists("base.pt"):
    ck = torch.load("base.pt", map_location=DEV)
    saved = ck.get("meta")
    if saved is None:
        print("WARNING: base.pt carries no setup metadata (older run); split and schedule "
              "cannot be verified. Treat any comparison against it with care.")
    else:
        mismatch = {k: (saved.get(k), v) for k, v in CKPT_META.items() if saved.get(k) != v}
        assert not mismatch, f"base.pt was trained under a different setup: {mismatch}"
    net.load_state_dict(ck["state"])
    base_hist = [{"epoch": ck["epoch"], "train": float("nan"), "val": ck["val"], "best": True}]
    print(f"loaded base.pt (epoch {ck['epoch']}, val {ck['val']:.4f}); skipping base training")
else:
    base_hist = run_epochs(net, list(net.parameters()), cfg.base_epochs, cfg.lr_base,
                           "base", base_train, base_val, "base.pt")
    ck = torch.load("base.pt", map_location=DEV); ck["meta"] = CKPT_META; torch.save(ck, "base.pt")

## Attach cross-attention and train only that block

The backbone is frozen. One `TextCrossAttention` block is inserted after the stage named
by `cfg.attach`; it flattens the feature map into query tokens, attends to the text tokens,
and adds the result back. Its output projection is zero-initialised (PixArt-α's own
initialisation), so the first assertion — adapted model equals base at step zero — is what
makes this a controlled comparison.

`COND(y)` is the label the backbone sees: the true label if `cfg.label_path`, otherwise the
null token. With the null token, the text is the model's only knowledge of the class.

In [ ]:
ckpt = torch.load("base.pt", map_location=DEV)
base_frozen = make_backbone().to(DEV)
base_frozen.load_state_dict(ckpt["state"])
base_frozen.eval()

def COND(y):
    '''The label the frozen backbone sees. Null by default, so the class can only reach the
    model through the text. cfg.label_path = True gives the backbone the true label.'''
    return y if cfg.label_path else torch.full_like(y, NULL_ID)

probe_x, probe_y = xb[:4].to(DEV), cb[:4].to(DEV)
probe_t = torch.rand(4, device=DEV)
with torch.no_grad():
    ref = base_frozen(probe_x, probe_t, COND(probe_y)).clone()

# PixArtBlock.forward: x = x + cross_attn(x, text), here after the U-Net stage named by
# cfg.attach. Queries come from the feature map, keys and values from the text tokens.
xa = TextConditioned(base_frozen, attach=cfg.attach, channels=MID_CHANNELS,
                     text_dim=TEXT_DIM, heads=cfg.heads).to(DEV)

# PixArt.initialize_weights: the output projection starts at zero, so the block is the
# identity at this point and everything it later does is measured from the base model.
with torch.no_grad():
    got = xa(probe_x, probe_t, TEXT_TOK[probe_y], TEXT_MASK[probe_y], cond_ids=COND(probe_y))
assert torch.allclose(got, ref, atol=1e-5), (got - ref).abs().max().item()
print("no-op at init confirmed")

assert all(".attn." in n for n, p in xa.named_parameters() if p.requires_grad)
# The adapter now lives INSIDE base_frozen, so count frozen params by requires_grad, not
# by module membership -- otherwise the adapter is counted on both sides.
N_FROZEN = sum(p.numel() for p in base_frozen.parameters() if not p.requires_grad)
print(f"trainable: {xa.n_trainable():,} params, all in the cross-attention block")
print(f"frozen:    {N_FROZEN:,} params")
print(f"label path: {'ON (true label)' if cfg.label_path else 'OFF (null label; text only)'}")

In [ ]:
def xa_train(x, y):
    # The ordinary noise-prediction loss. Only the attention block has requires_grad; the
    # U-Net and CLIP stay fixed. No guidance anywhere: the text acts only through attention.
    xk, k, eps = add_noise(x)
    return F.mse_loss(xa(xk, k.float() / cfg.K, TEXT_TOK[y], TEXT_MASK[y], cond_ids=COND(y)), eps)


def xa_val(x, y, gen):
    xk, k, eps = add_noise(x, gen)
    return F.mse_loss(xa(xk, k.float() / cfg.K, TEXT_TOK[y], TEXT_MASK[y], cond_ids=COND(y)), eps)


xa_hist = run_epochs(xa, [p for p in xa.parameters() if p.requires_grad],
                     cfg.xattn_epochs, cfg.lr_xattn, "xattn", xa_train, xa_val, "xattn.pt")

best = torch.load("xattn.pt", map_location=DEV)["state"]
xa.load_state_dict(best)
xa.eval()
print("restored best cross-attention block")

## The comparable numbers, and the diagnostic that matters

Three validation MSEs on identical examples, timesteps and noise:

- **base, same label path** — what the text-conditioned model starts from. With the label
  path off this is the *unconditional* model.
- **base, true label** — a reference: what knowing the class is worth when it comes in
  through the label embedding.
- **cross-attention** — the arm under test.

And the diagnostic: validation with the **wrong text**. Labels are shifted by a fixed
offset, so every example gets a genuinely different class, and the shift consumes no
random state, so correct- and wrong-text passes see byte-identical noise and timesteps.
(A within-batch permutation would fail both tests here: the validation set is ordered by
class, so most permuted labels would land on the same class, and drawing the permutation
from the noise generator would desynchronise the two passes.) Averaged over three offsets.

Read the result as *average loss sensitivity to the text*. A small aggregate difference
is a finding, not proof the text is ignored — it could matter only at some noise levels,
or only for the samples, which the next cells check separately. A null here is
reportable. Base-vs-adapter differences include the adaptation itself; correct-vs-wrong
text *within* the adapted model is what isolates the text.

In [ ]:
def frozen_val_same_path(x, y, gen):
    xk, k, eps = add_noise(x, gen)
    return F.mse_loss(base_frozen(xk, k.float() / cfg.K, COND(y)), eps)


def frozen_val_label(x, y, gen):
    xk, k, eps = add_noise(x, gen)
    return F.mse_loss(base_frozen(xk, k.float() / cfg.K, y), eps)


def xa_val_wrong_text(offset):
    '''Wrong text via a fixed label shift: always a different class, never touches gen.
    Only the text changes. Noise, timesteps and the backbone's label input are identical to
    the correct-text pass, so any loss difference is the text.'''
    def fn(x, y, gen):
        xk, k, eps = add_noise(x, gen)
        y_wrong = (y + offset) % meta["n_conditions"]
        return F.mse_loss(xa(xk, k.float() / cfg.K, TEXT_TOK[y_wrong], TEXT_MASK[y_wrong],
                             cond_ids=COND(y)), eps)
    return fn


WRONG_OFFSETS = (1, 37, 73)

VAL_BASE_SAME = validate(base_frozen, frozen_val_same_path)
VAL_BASE_LABEL = validate(base_frozen, frozen_val_label)
VAL_XA = validate(xa, xa_val)
VAL_XA_WRONG_EACH = [validate(xa, xa_val_wrong_text(o)) for o in WRONG_OFFSETS]
VAL_XA_WRONG = sum(VAL_XA_WRONG_EACH) / len(VAL_XA_WRONG_EACH)

print(f"base, same label path as xattn : {VAL_BASE_SAME:.4f}")
print(f"base, true label (reference)   : {VAL_BASE_LABEL:.4f}")
print(f"xattn, correct text            : {VAL_XA:.4f}   ({(VAL_BASE_SAME - VAL_XA) / VAL_BASE_SAME:+.1%} vs base)")
print(f"xattn, wrong text (mean of {len(WRONG_OFFSETS)}) : {VAL_XA_WRONG:.4f}   "
      f"per offset {[round(v, 4) for v in VAL_XA_WRONG_EACH]}")
TEXT_EFFECT = (VAL_XA_WRONG - VAL_XA) / VAL_XA
print()
print(f"text effect (wrong - correct) / correct = {TEXT_EFFECT:+.2%}")
if abs(TEXT_EFFECT) < 0.005:
    print("little average loss sensitivity to the text at this training budget. "
          "Check the per-noise-level breakdown and the sample comparison below before concluding.")

## Text sensitivity by noise level

The aggregate number above averages over the whole trajectory. Text may matter only late
(fine detail) or only early (layout). This bins the correct-vs-wrong loss gap by noise
level on the same fixed noise.

In [ ]:
@torch.no_grad()
def loss_by_noise_level(offset, n_bins=5):
    xa.eval()
    gen = torch.Generator(device=DEV).manual_seed(VAL_SEED)
    edges = torch.linspace(0, cfg.K, n_bins + 1, device=DEV)
    sums = torch.zeros(n_bins, 2, device=DEV); counts = torch.zeros(n_bins, device=DEV)
    for x, y in val_loader:
        x, y = x.to(DEV), y.to(DEV)
        xk, k, eps = add_noise(x, gen)
        t = k.float() / cfg.K
        yw = (y + offset) % meta["n_conditions"]
        e_ok = ((xa(xk, t, TEXT_TOK[y], TEXT_MASK[y], cond_ids=COND(y)) - eps) ** 2).flatten(1).mean(1)
        e_wr = ((xa(xk, t, TEXT_TOK[yw], TEXT_MASK[yw], cond_ids=COND(y)) - eps) ** 2).flatten(1).mean(1)
        b = torch.bucketize(k.float(), edges[1:-1])
        sums.index_add_(0, b, torch.stack([e_ok, e_wr], 1)); counts.index_add_(0, b, torch.ones_like(b, dtype=torch.float))
    return edges.cpu(), (sums / counts[:, None]).cpu()

edges, per_bin = loss_by_noise_level(WRONG_OFFSETS[0])
print(f"{'noise level (k)':>18} {'correct':>9} {'wrong':>9} {'gap':>8}")
TEXT_EFFECT_BY_BIN = []
for i in range(len(per_bin)):
    ok, wr = per_bin[i].tolist()
    TEXT_EFFECT_BY_BIN.append((wr - ok) / ok)
    print(f"{int(edges[i]):>7}-{int(edges[i+1]):<9} {ok:>9.4f} {wr:>9.4f} {(wr-ok)/ok:>+8.2%}")

## Training curves

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for a, (name, hist) in zip(ax, [("base", base_hist), ("cross-attention", xa_hist)]):
    ep = [h["epoch"] for h in hist]
    if len(hist) > 1:
        a.plot(ep, [h["train"] for h in hist], label="train")
    a.plot(ep, [h["val"] for h in hist], label="val", marker="o" if len(hist) == 1 else None)
    bst = min(hist, key=lambda h: h["val"])
    a.axvline(bst["epoch"], ls="--", c="grey", lw=1)
    a.scatter([bst["epoch"]], [bst["val"]], c="red", zorder=5,
              label=f"selected (epoch {bst['epoch']})")
    a.set_title(f"{name}: best val {bst['val']:.4f}")
    a.set_xlabel("epoch"); a.set_ylabel("noise-prediction MSE"); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Samples — same starting noise, three conditions

Row 1: the base model, same label path as the arm under test. Row 2: cross-attention with
the **correct** description. Row 3: cross-attention with a **wrong** description (labels
shifted by one). All three start from the same noise, so any difference is the text.

In [ ]:
N = 8
y_eval = torch.arange(N, device=DEV) % meta["n_conditions"]
y_wrong = (y_eval + 1) % meta["n_conditions"]
g = torch.Generator(device=DEV).manual_seed(cfg.seed + 1)
X0 = torch.randn(N, *meta["shape"], device=DEV, generator=g)

def paired_gen():
    '''Fresh generator, same seed: identical per-step noise for every row of a paired comparison.'''
    return torch.Generator(device=DEV).manual_seed(cfg.seed + 2)

base_s = ddim_sample(lambda x, t: base_frozen(x, t, COND(y_eval)), N, meta["shape"],
                     cfg.sample_steps, x_init=X0, generator=paired_gen())
xa_s = ddim_sample(lambda x, t: xa(x, t, TEXT_TOK[y_eval], TEXT_MASK[y_eval], cond_ids=COND(y_eval)),
                   N, meta["shape"], cfg.sample_steps, x_init=X0, generator=paired_gen())
xa_wrong = ddim_sample(lambda x, t: xa(x, t, TEXT_TOK[y_wrong], TEXT_MASK[y_wrong], cond_ids=COND(y_eval)),
                       N, meta["shape"], cfg.sample_steps, x_init=X0, generator=paired_gen())
print("sampled:", tuple(base_s.shape), "| NFE per sample:", cfg.sample_steps,
      "| paired: same X0 for all rows")

def short(i):
    return meta["cond_text"][int(i)].replace("a photo of a ", "")

fig, axes = plt.subplots(3, N, figsize=(2 * N, 6.6))
rows = [("base", base_s, y_eval), ("xattn, correct text", xa_s, y_eval),
        ("xattn, WRONG text", xa_wrong, y_wrong)]
for r, (name, s, ys) in enumerate(rows):
    for j in range(N):
        axes[r, j].imshow(((s[j].cpu() + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy())
        axes[r, j].set_xticks([]); axes[r, j].set_yticks([])
        if r == 1:
            axes[r, j].set_title(short(ys[j]), fontsize=8)
        if r == 2:
            axes[r, j].set_xlabel(short(ys[j]), fontsize=7)
    axes[r, 0].set_ylabel(name, fontsize=10)
plt.tight_layout(); plt.show()

SAMPLE_TEXT_DIFF = ((xa_s - xa_wrong).flatten(1).norm(dim=1) / xa_s.flatten(1).norm(dim=1)).mean().item()
print(f"mean relative difference between correct-text and wrong-text samples: {SAMPLE_TEXT_DIFF:.1%}")

## Denoising trajectory

Four samples along the reverse process. The first column is the starting noise; the rest
show the model's **clean-image estimate** $\hat x_0$ at evenly spaced steps, labelled by
timestep $k$, ending at the finished sample. The noisy state $x_t$ itself looks like noise
until the last few steps, which is why it is not what is shown.

In [ ]:
traj_y = y_eval[:4]
_, snaps = ddim_sample(lambda x, t: xa(x, t, TEXT_TOK[traj_y], TEXT_MASK[traj_y], cond_ids=COND(traj_y)),
                       4, meta["shape"], cfg.sample_steps, n_snapshots=6)

fig, axes = plt.subplots(4, len(snaps), figsize=(1.6 * len(snaps), 6.6))
for r in range(4):
    for c, (k, s) in enumerate(snaps):
        axes[r, c].imshow(((s[r] + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy())
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
        if r == 0:
            axes[r, c].set_title("noise $x_T$" if c == 0 else
                                 ("sample" if c == len(snaps) - 1 else f"$\\hat x_0$, k={k}"), fontsize=8)
    axes[r, 0].set_ylabel(short(traj_y[r]), fontsize=7)
plt.tight_layout(); plt.show()

## FID (optional, needs a GPU)

Set `n_fid = 10000` for a reportable number. Real images are the 5000-image validation
split; record both counts next to the number.

In [ ]:
n_fid = int(os.environ.get("N_FID", 0))   # 10000 for a reportable number; 0 skips this cell. Headless: N_FID=10000

FID_SCORES = {}

if n_fid:
    import importlib.util, subprocess, sys
    for pkg, mod in [("torchmetrics", "torchmetrics"), ("torch-fidelity", "torch_fidelity")]:
        if importlib.util.find_spec(mod) is None:
            print(f"installing {pkg} ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

    from torchmetrics.image.fid import FrechetInceptionDistance

    def to_uint8(x):
        return ((x.clamp(-1, 1) + 1) * 127.5).round().to(torch.uint8)

    def sample_for(which, y):
        if which == "base":
            return ddim_sample(lambda x, t: base_frozen(x, t, COND(y)),
                               len(y), meta["shape"], cfg.sample_steps)
        return ddim_sample(lambda x, t: xa(x, t, TEXT_TOK[y], TEXT_MASK[y], cond_ids=COND(y)),
                           len(y), meta["shape"], cfg.sample_steps)

    for which in ("base", "xattn"):
        fid = FrechetInceptionDistance(feature=2048, normalize=False).to(DEV)
        n_real = 0
        for x, _ in val_loader:
            fid.update(to_uint8(x.to(DEV)), real=True)
            n_real += len(x)
        torch.manual_seed(cfg.seed)
        for start in range(0, n_fid, 100):
            y = torch.arange(start, min(start + 100, n_fid), device=DEV) % meta["n_conditions"]
            fid.update(to_uint8(sample_for(which, y)), real=False)
        FID_SCORES[f"{which}_fid"] = float(fid.compute())
        FID_SCORES["fid_n_real"] = n_real
        FID_SCORES["fid_n_generated"] = n_fid
        print(f"{which} FID {FID_SCORES[f'{which}_fid']:.2f}   "
              f"(real {n_real}, generated {n_fid}, NFE {cfg.sample_steps})")

    if min(FID_SCORES["fid_n_real"], n_fid) < 10000:
        print()
        print("NOTE: FID is biased upward below ~10k samples. Report both counts.")
else:
    print("FID skipped (n_fid = 0)")

## Manifest

In [ ]:
report = {
    "method": "text conditioning by cross-attention (PixArt-alpha block, arXiv:2310.00426)",
    "modality": "images",
    "code_version": GIT_SHA,
    "config": asdict(cfg),
    "label_path": cfg.label_path,
    "val_matched_base_same_path": VAL_BASE_SAME,
    "val_matched_base_true_label": VAL_BASE_LABEL,
    "val_matched_xattn": VAL_XA,
    "val_matched_xattn_wrong_text": VAL_XA_WRONG,
    "val_matched_xattn_wrong_text_per_offset": dict(zip(map(str, WRONG_OFFSETS), VAL_XA_WRONG_EACH)),
    "text_effect_rel": TEXT_EFFECT,
    "text_effect_rel_by_noise_bin": TEXT_EFFECT_BY_BIN,
    "sample_text_diff_rel": SAMPLE_TEXT_DIFF,
    "base_best_val_during_training": min(h["val"] for h in base_hist),
    "xattn_best_val_during_training": min(h["val"] for h in xa_hist),
    "trainable_xattn": xa.n_trainable(),
    "frozen_backbone": N_FROZEN,
    "checkpoint_meta": CKPT_META,
    "attach": cfg.attach,
    "nfe": cfg.sample_steps,
    "clip_fallback": IS_FALLBACK,
    "train_seconds": {"base": sum(h.get("seconds", 0) for h in base_hist),
                      "xattn": sum(h.get("seconds", 0) for h in xa_hist)},
    "hardware": torch.cuda.get_device_name(0) if DEV.type == "cuda" else "cpu",
    "device": str(DEV),
}
report.update(globals().get("FID_SCORES", {}))
print(json.dumps(report, indent=2))
with open("manifest_xattn_images.json", "w") as f:
    json.dump(report, f, indent=2)

## Before reporting any of this

- [ ] `clip_fallback: false` in the manifest
- [ ] `text_effect_rel` reported with its sign and the per-noise-level breakdown; a small
      or null effect is a reportable result, not a reason to withhold the row
- [ ] FID with `n_fid = 10000`; FID measures image quality, not whether the text was used
- [ ] Three seeds (`seed`), mean +/- std, with `split_seed` held fixed
- [ ] `label_path` stated next to every number; the two settings are different experiments
- [ ] Described as **a frozen U-Net with a PixArt-α-inspired residual cross-attention
      adapter** — CLIP not T5, one block not every block, attention-only training, no CFG
      — not as PixArt-α reproduced